# Device Management and GPU Usage

Hardware mistakes in PyTorch are usually small, local, and expensive. Accidentally moving tensors back to CPU, choosing a batch size that underuses the GPU, or calling synchronizing operations too often can wipe out the speedup you expected.

This notebook stays close to native PyTorch and focuses on three habits:

1. Choose and inspect the device explicitly.
2. Move tensors in one direction, at well-defined points.
3. Measure throughput before and after changing batch size or precision.


In [1]:
import time

import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

def choose_device(preferred_index=0):
    if torch.cuda.is_available():
        count = torch.cuda.device_count()
        idx = min(preferred_index, count - 1)
        return torch.device(f'cuda:{idx}')
    return torch.device('cpu')

device = choose_device()
print(f'Using device: {device}')
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(f'cuda:{idx} | {props.name} | {props.total_memory / 1024**3:.1f} GB')
else:
    print('No CUDA device found. The notebook still demonstrates the code structure, but GPU-specific differences will not appear.')


Using device: cuda:0
cuda:0 | NVIDIA RTX 6000 Ada Generation | 47.5 GB
cuda:1 | NVIDIA RTX 6000 Ada Generation | 47.5 GB
cuda:2 | NVIDIA RTX 6000 Ada Generation | 47.5 GB
cuda:3 | NVIDIA RTX 6000 Ada Generation | 47.5 GB
cuda:4 | NVIDIA RTX 6000 Ada Generation | 47.5 GB
cuda:5 | NVIDIA RTX 6000 Ada Generation | 47.5 GB
cuda:6 | NVIDIA RTX 6000 Ada Generation | 47.5 GB
cuda:7 | NVIDIA RTX 6000 Ada Generation | 47.5 GB


In [2]:
n_samples = 8192
n_features = 256
n_classes = 4

X = torch.randn(n_samples, n_features)
W = torch.randn(n_features, n_classes)
logits = X @ W + 0.2 * torch.randn(n_samples, n_classes)
y = logits.argmax(dim=1)

dataset = TensorDataset(X, y)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):
        return self.net(x)


## Batch-Size Sweep

Throughput almost always changes with batch size. On a GPU, very small batches often underutilize the hardware. On a CPU, the relationship can be different, so it is still worth measuring.


In [3]:
def benchmark_epoch(batch_size, use_autocast=False):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=torch.cuda.is_available())
    model = MLP().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    amp_enabled = use_autocast and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)

    start = time.perf_counter()
    total = 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            pred = model(xb)
            loss = criterion(pred, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += xb.size(0)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    return {
        'batch_size': batch_size,
        'autocast': amp_enabled,
        'seconds': elapsed,
        'samples_per_second': total / elapsed,
    }

rows = []
for batch_size in [32, 64, 128, 256, 512]:
    rows.append(benchmark_epoch(batch_size, use_autocast=False))
    rows.append(benchmark_epoch(batch_size, use_autocast=True))

batch_results = pd.DataFrame(rows)
batch_results.sort_values('samples_per_second', ascending=False)


,batch_size,autocast,seconds,samples_per_second
4,128,False,0.108527,75483.206024
5,128,True,0.139338,58792.260281
2,64,False,0.182706,44837.028109
8,512,False,0.231776,35344.422946
9,512,True,0.235458,34791.754102
3,64,True,0.245151,33416.143744
6,256,False,0.425088,19271.303676
7,256,True,0.451133,18158.707203
1,32,True,0.617786,13260.249942
0,32,False,0.638446,12831.154238


## Matrix Multiplication: CPU vs GPU

A GPU is not automatically faster for every operation. Small matrix multiplications can be dominated by launch overhead, while larger matrix multiplications usually give the GPU enough work to use many cores at once. The benchmark below keeps tensors on each device so we are timing the matrix multiply, not the cost of copying data to the GPU.


In [8]:
def time_matmul(n, repeats, run_device):
    a = torch.randn(n, n, device=run_device)
    b = torch.randn(n, n, device=run_device)

    with torch.no_grad():
        for _ in range(2):
            _ = a @ b
        if run_device.type == 'cuda':
            torch.cuda.synchronize(run_device)

        times = []
        for _ in range(repeats):
            start = time.perf_counter()
            _ = a @ b
            if run_device.type == 'cuda':
                torch.cuda.synchronize(run_device)
            times.append(time.perf_counter() - start)

    median_seconds = sorted(times)[len(times) // 2]
    gflops = (2 * n**3) / median_seconds / 1e9
    return median_seconds, gflops


matrix_sizes = [128, 256, 512, 1024, 2048, 4096, 8192, 16384]
matmul_rows = []
repeats = 10

for n in matrix_sizes:
    cpu_seconds, cpu_gflops = time_matmul(n, repeats, torch.device('cpu'))
    row = {
        'n': n,
        'cpu_seconds': cpu_seconds,
        'cpu_gflops': cpu_gflops,
    }

    if torch.cuda.is_available():
        gpu_seconds, gpu_gflops = time_matmul(n, repeats, device)
        row.update({
            'gpu_seconds': gpu_seconds,
            'gpu_gflops': gpu_gflops,
            'gpu_speedup': cpu_seconds / gpu_seconds,
        })

    matmul_rows.append(row)

matmul_results = pd.DataFrame(matmul_rows)
matmul_results


,n,cpu_seconds,cpu_gflops,gpu_seconds,gpu_gflops,gpu_speedup
0,128,0.026067,0.160907,0.000023,185.424886,1152.371212
1,256,0.026368,1.272527,0.000030,1111.725408,873.636139
2,512,0.028830,9.311025,0.000032,8380.739013,900.087695
3,1024,0.026847,79.988760,0.000092,23435.269221,292.982031
4,2048,0.031119,552.073327,0.000369,46528.169200,84.278966
5,4096,0.058188,2361.996524,0.002806,48986.488692,20.739441
6,8192,0.482539,2278.597710,0.033551,32771.475565,14.382300
7,16384,2.291008,3839.398607,0.294581,29859.671969,7.777174


## A typical training loop

This example shows a typical training loop with some tricks that can be commonly used.

In [12]:
def epoch(batch_size=256, train=True):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, pin_memory=torch.cuda.is_available())
    model = MLP().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=5e-2)
    criterion = nn.CrossEntropyLoss()
    losses = []
    preds = []
    start = time.perf_counter()

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                pred = model(xb)
                loss = criterion(pred, yb)

        losses.append(loss.item())
        preds.append(pred.detach().cpu().numpy())

    if device.type == 'cuda':
        torch.cuda.synchronize()
    return time.perf_counter() - start, sum(losses) / len(losses)

train_seconds, train_loss = epoch(train=True)
test_seconds, test_loss = epoch(train=False)

print(f'Training epoch took {train_seconds:.2f} seconds, average loss: {train_loss:.4f}')
print(f'Testing epoch took {test_seconds:.2f} seconds, average loss: {test_loss:.4f}')

Training epoch took 0.44 seconds, average loss: 1.3792
Testing epoch took 0.43 seconds, average loss: 1.3935


In [13]:
if device.type == 'cuda':
    allocated = torch.cuda.memory_allocated(device) / 1024**2
    reserved = torch.cuda.memory_reserved(device) / 1024**2
    print(f'Memory allocated: {allocated:.1f} MiB')
    print(f'Memory reserved:  {reserved:.1f} MiB')
else:
    print('CUDA memory statistics are only available when running on a GPU.')


Memory allocated: 17.2 MiB
Memory reserved:  30.0 MiB


**Device management checklist**

1. Pick the device explicitly rather than relying on hidden defaults.
2. Move model and tensors once per batch, not back and forth inside helper functions.
3. Sweep batch size before claiming a GPU is slow.
4. Use mixed precision only after confirming your device supports it and your loss stays stable.
